# RSM338 Assignment 6 Part B: Regression
## April 8, 2026
### Ethan Wang, Kevin Yang

In [2]:
import numpy as np
import pandas as pd
from scipy.stats import norm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch

# Define the Black-Scholes function for Call Options
def black_scholes_call(S, K, T, r, sigma):
    # Calculate d1 and d2
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    
    # Calculate call price
    call_price = S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    return call_price

# Set seed for reproducibility
np.random.seed(23)
n_samples = 100000

# Sample inputs uniformly from the specified ranges
S0 = np.random.uniform(40, 60, n_samples)
# K depends on S0, so we sample it relative to the S0 array
K = np.random.uniform(0.5 * S0, 1.5 * S0, n_samples) 
T = np.random.uniform(0.25, 2.0, n_samples)
r = np.random.uniform(0, 0.05, n_samples)
sigma = np.random.uniform(0.10, 0.40, n_samples)

# Calculate the true Black-Scholes price for all 100,000 contracts simultaneously
C = black_scholes_call(S0, K, T, r, sigma)

# Combine into a DataFrame
df_bs = pd.DataFrame({
    'S0': S0, 'K': K, 'T': T, 'r': r, 'sigma': sigma, 'Call_Price': C
})

print(f"Generated {len(df_bs)} Black-Scholes samples.")
df_bs.head()

Generated 100000 Black-Scholes samples.


,S0,K,T,r,sigma,Call_Price
0,50.345958,33.297847,1.671844,0.020493,0.201643,18.352483
1,58.939252,79.685795,0.889926,0.020344,0.130935,0.031118
2,55.309195,37.551077,1.241290,0.008531,0.206519,18.330442
3,45.647917,62.834540,0.796120,0.039218,0.319233,1.214502
4,44.420907,30.589633,1.437705,0.010145,0.381525,16.095141


In [3]:
# Separate features (X) and target (y)
X = df_bs[['S0', 'K', 'T', 'r', 'sigma']].values
y = df_bs['Call_Price'].values.reshape(-1, 1) # Reshape for neural network target

# 1. First split: 20% to Test, 80% to Temp
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

# 2. Second split: 25% of Temp (which is 20% of total) to Val, rest to Train
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42)

# Standardize the inputs based ONLY on the training set
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Convert all splits to PyTorch FloatTensors
X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)

X_val_t = torch.tensor(X_val_scaled, dtype=torch.float32)
y_val_t = torch.tensor(y_val, dtype=torch.float32)

X_test_t = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32)

print(f"Train shapes: X={X_train_t.shape}, y={y_train_t.shape}")
print(f"Val shapes: X={X_val_t.shape}, y={y_val_t.shape}")
print(f"Test shapes: X={X_test_t.shape}, y={y_test_t.shape}")

Train shapes: X=torch.Size([60000, 5]), y=torch.Size([60000, 1])
Val shapes: X=torch.Size([20000, 5]), y=torch.Size([20000, 1])
Test shapes: X=torch.Size([20000, 5]), y=torch.Size([20000, 1])


In [4]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# 1. Initialize and fit the Linear Regression model on the training set
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)

# 2. Generate predictions on the unseen test set
lr_predictions = lr_model.predict(X_test_scaled)

# 3. Calculate Mean Squared Error (MSE) and R-squared (R^2)
lr_mse = mean_squared_error(y_test, lr_predictions)
lr_r2 = r2_score(y_test, lr_predictions)

print(f"Linear Regression Test MSE: {lr_mse:.4f}")
print(f"Linear Regression Test R^2: {lr_r2:.4f}")

Linear Regression Test MSE: 6.0794
Linear Regression Test R^2: 0.8991


The results of the linear regression appear to be promising at first. The R-squared is quite high, at 0.899. However, the mean-squared error is 6.08, which is very high in our case. Essentially, the linear regression is able to capture the general direction of the relationship, likely due to strong feature relationships like stock price, but it does not help predict call option prices because as we know, the Black-Scholes equation is quite non-linear. 